# SustAInable — Notebook 02: Feature Engineering & Label Construction
**Project:** SustAInable — Neighborhood Heat Illness Risk Prediction by ZIP Code  
**Author:** Nico Meyering, MPA | Equitech Futures CTI 2026  
**GitHub:** [github.com/meyeringn/sustainable-heat](https://github.com/meyeringn/sustainable-heat)

---

## What This Notebook Does

Raw data rarely arrives ready to train a model. This notebook engineers meaningful features from the raw ZIP-level dataset, constructs the binary label, and prepares the full feature matrix for modeling.

**Key engineering decisions:**
- **Composite vulnerability score** — combines poverty, age, and health baseline into a single index
- **Infrastructure gap index** — captures the distance between heat exposure and cooling access
- **Interaction terms** — extreme heat × low AC access captures the compounding risk that raw features miss individually
- **Label:** Binary — 1 = elevated heat illness risk ZIP code, 0 = lower risk

In [ ]:
# !pip install xgboost imbalanced-learn shap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid')
print('Libraries loaded.')

## 1. Load Data

In [ ]:
df = pd.read_csv('sustainable_zip_data.csv')
print(f'Loaded: {df.shape}')
print(f'Elevated risk rate: {df["elevated_heat_risk"].mean():.1%}')

## 2. Feature Engineering

In [ ]:
df_feat = df.copy()

# Composite social vulnerability index (0–1 scale)
scaler = MinMaxScaler()
vuln_components = df_feat[['poverty_rate_pct','pct_65_plus','uninsured_rate_pct',
                            'prev_heart_disease_pct','prev_diabetes_pct']].values
df_feat['social_vulnerability_index'] = scaler.fit_transform(vuln_components).mean(axis=1)

# Infrastructure gap: high heat exposure, low cooling access
df_feat['infrastructure_gap'] = (
    (df_feat['extreme_heat_days_annual'] / df_feat['extreme_heat_days_annual'].max()) -
    (df_feat['ac_access_pct'] / 100) -
    (df_feat['cooling_centers_per_10k'] / df_feat['cooling_centers_per_10k'].max())
)

# Heat × vulnerability interaction
df_feat['heat_x_vulnerability'] = (
    df_feat['avg_summer_temp_f'] / 100 * df_feat['social_vulnerability_index']
)

# Canopy deficit (impervious surface relative to tree cover)
df_feat['canopy_deficit'] = df_feat['impervious_surface_pct'] - df_feat['tree_canopy_coverage_pct']

# Encode categoricals
le_region = LabelEncoder()
le_urban = LabelEncoder()
df_feat['region_encoded'] = le_region.fit_transform(df_feat['region'])
df_feat['urban_type_encoded'] = le_urban.fit_transform(df_feat['urban_type'])
joblib.dump({'region': le_region, 'urban_type': le_urban}, 'sustainable_encoders.pkl')

print('Engineered features added:')
print('  social_vulnerability_index')
print('  infrastructure_gap')
print('  heat_x_vulnerability')
print('  canopy_deficit')
print(f'\nDataset shape: {df_feat.shape}')

## 3. Define Feature Set & Split

In [ ]:
FEATURE_COLS = [
    'avg_summer_temp_f', 'extreme_heat_days_annual', 'urban_heat_island_intensity',
    'avg_humidity_pct', 'tree_canopy_coverage_pct', 'impervious_surface_pct',
    'cooling_centers_per_10k', 'ac_access_pct', 'poverty_rate_pct',
    'pct_65_plus', 'pct_outdoor_workers', 'median_household_income_k',
    'uninsured_rate_pct', 'prev_heart_disease_pct', 'prev_diabetes_pct',
    'prev_obesity_pct', 'prev_copd_pct', 'heat_hazard_index',
    'social_vulnerability_index', 'infrastructure_gap',
    'heat_x_vulnerability', 'canopy_deficit',
    'region_encoded', 'urban_type_encoded'
]
TARGET = 'elevated_heat_risk'

X = df_feat[FEATURE_COLS]
y = df_feat[TARGET]

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {X_train.shape} | Holdout: {X_holdout.shape}')
print(f'Train class balance: {y_train.value_counts().to_dict()}')

## 4. SMOTE Balancing

In [ ]:
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f'After SMOTE: {X_train_bal.shape}')
print(f'New balance: {pd.Series(y_train_bal).value_counts().to_dict()}')

# Save
np.save('sustainable_X_train.npy', X_train_bal)
np.save('sustainable_y_train.npy', y_train_bal)
np.save('sustainable_X_holdout.npy', X_holdout.values)
np.save('sustainable_y_holdout.npy', y_holdout.values)
joblib.dump(FEATURE_COLS, 'sustainable_feature_cols.pkl')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('SustAInable — Class Balance Before and After SMOTE', fontweight='bold')
labels = ['Lower Risk (0)', 'Elevated Risk (1)']
before = y_train.value_counts().sort_index()
after = pd.Series(y_train_bal).value_counts().sort_index()
for ax, data, title in zip(axes, [before, after], ['Before SMOTE', 'After SMOTE (Training Only)']):
    ax.bar(labels, data.values, color=['#4CAF50','#E53935'], alpha=0.85, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Record Count')
    for i, v in enumerate(data.values):
        ax.text(i, v+10, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('sustainable_smote_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sustainable_smote_balance.png')
print('\nAll arrays saved. → Notebook 03: Label Construction & Validation')